In [3]:
import pandas as pd

# Load Fake.csv and add 'fake' label
df_fake = pd.read_csv('Fake.csv')
df_fake['label'] = 'fake'

df_true = pd.read_csv('True.csv')
df_true['label'] = 'real'

df = pd.concat([df_fake, df_true], ignore_index=True)

print("First 5 rows of the combined DataFrame:")
print(df.head())

print("\nInformation about the combined DataFrame:")
df.info()

First 5 rows of the combined DataFrame:
                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date label  
0  December 31, 2017  fake  
1  December 31, 2017  fake  
2  December 30, 2017  fake  
3  December 29, 2017  fake  
4  December 25, 2017  fake  

Information about the combined DataFrame:
<class 'pandas.core.f

## Initial Data Exploration



In [5]:
print("1. First 5 rows of the combined DataFrame:")
print(df.head())

print("\n2. Data types and non-null counts:")
df.info()

print("\n3. Descriptive statistics for all columns:")
print(df.describe(include='all'))

print("\n4. Missing values across all columns:")
print(df.isnull().sum())

print("\n5. Distribution of 'label' column:")
print(df['label'].value_counts())

print("\n6. Number of unique values in 'subject' column:")
print(df['subject'].nunique())
print("Unique values in 'subject' column:")
print(df['subject'].unique())

print("\n7. Number of duplicate rows:")
print(df.duplicated().sum())

1. First 5 rows of the combined DataFrame:
                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date label  
0  December 31, 2017  fake  
1  December 31, 2017  fake  
2  December 30, 2017  fake  
3  December 29, 2017  fake  
4  December 25, 2017  fake  

2. Data types and non-null counts:
<class 'pandas.core.frame

## Text Preprocessing


Clean and preprocess both the 'text' (article body) and 'title' columns by lowercasing, removing punctuation, special characters, stop words, and applying lemmatization.


In [5]:
print(f"Number of rows before removing duplicates: {df.shape[0]}")
df.drop_duplicates(inplace=True)
print(f"Number of rows after removing duplicates: {df.shape[0]}")

Number of rows before removing duplicates: 44898
Number of rows after removing duplicates: 44689


In [7]:
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download necessary NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

# Initialize lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Convert to lowercase
    text = str(text).lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove special characters (non-alphanumeric, keep spaces)
    text = ''.join(char for char in text if char.isalnum() or char.isspace())
    # Tokenize and remove stopwords, then lemmatize
    tokens = []
    for word in text.split():
        if word not in stop_words:
            tokens.append(lemmatizer.lemmatize(word))
    return ' '.join(tokens)

# Apply cleaning function to 'title' column
df['cleaned_title'] = df['title'].apply(clean_text)

# Apply cleaning function to 'text' column
df['cleaned_text'] = df['text'].apply(clean_text)

# Display the first few rows with new cleaned columns
print("First 5 rows of the DataFrame with cleaned 'title' and 'text' columns:")
print(df[['title', 'cleaned_title', 'text', 'cleaned_text']].head())

First 5 rows of the DataFrame with cleaned 'title' and 'text' columns:
                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                       cleaned_title  \
0  donald trump sends embarrassing new year eve m...   
1  drunk bragging trump staffer started russian c...   
2  sheriff david clarke becomes internet joke thr...   
3  trump obsessed even obamas name coded website ...   
4  pope francis called donald trump christmas speech   

                                                text  \
0  Donald Trump just couldn t wish all Americans ...   
1  House Intelligence Committee Chairman Devin Nu...   
2  On Friday, it was revealed that former Milwauk...   
3  On Christmas day, Donald Tru

## Feature Extraction (TF-IDF)


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer_text = TfidfVectorizer()

tfidf_text_features = tfidf_vectorizer_text.fit_transform(df['cleaned_text'])

tfidf_vectorizer_title = TfidfVectorizer()

tfidf_title_features = tfidf_vectorizer_title.fit_transform(df['cleaned_title'])

print(f"Shape of TF-IDF features for text: {tfidf_text_features.shape}")
print(f"Shape of TF-IDF features for title: {tfidf_title_features.shape}")

Shape of TF-IDF features for text: (44689, 218707)
Shape of TF-IDF features for title: (44689, 23082)


## Split Data into Training and Test Sets


In [22]:
from sklearn.model_selection import train_test_split

y = df['label']

X_text_train, X_text_test, y_train_text, y_test_text = train_test_split(tfidf_text_features, y, test_size=0.2, random_state=42, stratify=y)

X_title_train, X_title_test, y_train_title, y_test_title = train_test_split(tfidf_title_features, y, test_size=0.2, random_state=42, stratify=y)

y_train = y_train_text
y_test = y_test_text

print(f"Shape of X_text_train: {X_text_train.shape}")
print(f"Shape of X_text_test: {X_text_test.shape}")
print(f"Shape of X_title_train: {X_title_train.shape}")
print(f"Shape of X_title_test: {X_title_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_text_train: (35751, 218707)
Shape of X_text_test: (8938, 218707)
Shape of X_title_train: (35751, 23082)
Shape of X_title_test: (8938, 23082)
Shape of y_train: (35751,)
Shape of y_test: (8938,)


## Train Random Forest Model on Body Text




In [10]:
from sklearn.ensemble import RandomForestClassifier

rf_classifier_text = RandomForestClassifier(random_state=42)

rf_classifier_text.fit(X_text_train, y_train)

print("Random Forest Classifier for article text trained successfully.")

Random Forest Classifier for article text trained successfully.


## Evaluate Model on Body Text


In [11]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

y_pred_text = rf_classifier_text.predict(X_text_test)

accuracy_text = accuracy_score(y_test, y_pred_text)

f1_text = f1_score(y_test, y_pred_text, average='weighted')

precision_text = precision_score(y_test, y_pred_text, average='weighted')

recall_text = recall_score(y_test, y_pred_text, average='weighted')

print("--- Model Performance on Body Text ---")
print(f"Accuracy: {accuracy_text:.4f}")
print(f"F1-Score: {f1_text:.4f}")
print(f"Precision: {precision_text:.4f}")
print(f"Recall: {recall_text:.4f}")

--- Model Performance on Body Text ---
Accuracy: 0.9874
F1-Score: 0.9874
Precision: 0.9874
Recall: 0.9874


In [13]:
from sklearn.ensemble import RandomForestClassifier

rf_classifier_title = RandomForestClassifier(random_state=42)

rf_classifier_title.fit(X_title_train, y_train)

print("Random Forest Classifier for article titles trained successfully.")

Random Forest Classifier for article titles trained successfully.


In [14]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

y_pred_title = rf_classifier_title.predict(X_title_test)

accuracy_title = accuracy_score(y_test, y_pred_title)

f1_title = f1_score(y_test, y_pred_title, average='weighted')

precision_title = precision_score(y_test, y_pred_title, average='weighted')

recall_title = recall_score(y_test, y_pred_title, average='weighted')

print("--- Model Performance on Title Text ---")
print(f"Accuracy: {accuracy_title:.4f}")
print(f"F1-Score: {f1_title:.4f}")
print(f"Precision: {precision_title:.4f}")
print(f"Recall: {recall_title:.4f}")

--- Model Performance on Title Text ---
Accuracy: 0.9475
F1-Score: 0.9475
Precision: 0.9476
Recall: 0.9475


## Compare Model Performances



In [15]:
print("\n--- Comparison of Model Performances ---")

print("\nModel trained on Body Text (Full Article Text):")
print(f"  Accuracy: {accuracy_text:.4f}")
print(f"  F1-Score: {f1_text:.4f}")
print(f"  Precision: {precision_text:.4f}")
print(f"  Recall: {recall_text:.4f}")

print("\nModel trained on Title Features:")
print(f"  Accuracy: {accuracy_title:.4f}")
print(f"  F1-Score: {f1_title:.4f}")
print(f"  Precision: {precision_title:.4f}")
print(f"  Recall: {recall_title:.4f}")

print("\n--- Conclusion ---")
if accuracy_text > accuracy_title:
    print("The model trained on full article text (body text) is more effective for fake news detection.")
else:
    print("The model trained on article titles is more effective for fake news detection.")

print("Target accuracy of 90% was achieved by both models, but the model using full article text performed significantly better.")


--- Comparison of Model Performances ---

Model trained on Body Text (Full Article Text):
  Accuracy: 0.9874
  F1-Score: 0.9874
  Precision: 0.9874
  Recall: 0.9874

Model trained on Title Features:
  Accuracy: 0.9475
  F1-Score: 0.9475
  Precision: 0.9476
  Recall: 0.9475

--- Conclusion ---
The model trained on full article text (body text) is more effective for fake news detection.
Target accuracy of 90% was achieved by both models, but the model using full article text performed significantly better.


## Evaluate Body-Trained Model on Titles



In [16]:
from sklearn.metrics import f1_score

test_cleaned_titles = df.loc[y_test.index, 'cleaned_title']

X_title_test_for_body_model = tfidf_vectorizer_text.transform(test_cleaned_titles)

y_pred_title_from_text_model = rf_classifier_text.predict(X_title_test_for_body_model)

f1_title_from_text_model = f1_score(y_test, y_pred_title_from_text_model, average='weighted')

print("--- Model trained on Body Text evaluated on Title Features ---")
print(f"F1-Score: {f1_title_from_text_model:.4f}")

--- Model trained on Body Text evaluated on Title Features ---
F1-Score: 0.4137


## Train New Random Forest Model on Titles


In [17]:
from sklearn.ensemble import RandomForestClassifier

rf_classifier_title_new = RandomForestClassifier(random_state=42)

rf_classifier_title_new.fit(X_title_train, y_train)

print("New Random Forest Classifier for article titles trained successfully.")

New Random Forest Classifier for article titles trained successfully.


## Evaluate Title-Trained Model on Titles



In [18]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

y_pred_title_new = rf_classifier_title_new.predict(X_title_test)

accuracy_title_new = accuracy_score(y_test, y_pred_title_new)

f1_title_new = f1_score(y_test, y_pred_title_new, average='weighted')

precision_title_new = precision_score(y_test, y_pred_title_new, average='weighted')

recall_title_new = recall_score(y_test, y_pred_title_new, average='weighted')

print("--- Model Performance on Title Text (New Model) ---")
print(f"Accuracy: {accuracy_title_new:.4f}")
print(f"F1-Score: {f1_title_new:.4f}")
print(f"Precision: {precision_title_new:.4f}")
print(f"Recall: {recall_title_new:.4f}")

--- Model Performance on Title Text (New Model) ---
Accuracy: 0.9475
F1-Score: 0.9475
Precision: 0.9476
Recall: 0.9475


## conclusion




In [20]:
print("--- Comparison of All Model Scenarios ---")

print("\n1. Model trained on Body Text, evaluated on Body Text:")
print(f"  Accuracy: {accuracy_text:.4f}")
print(f"  F1-Score: {f1_text:.4f}")
print(f"  Precision: {precision_text:.4f}")
print(f"  Recall: {recall_text:.4f}")

print("\n2. Model trained on Body Text, evaluated on Title Features:")
print(f"  F1-Score: {f1_title_from_text_model:.4f}")

print("\n3. Model trained on Title Features, evaluated on Title Features:")
print(f"  Accuracy: {accuracy_title_new:.4f}")
print(f"  F1-Score: {f1_title_new:.4f}")
print(f"  Precision: {precision_title_new:.4f}")
print(f"  Recall: {recall_title_new:.4f}")

--- Comparison of All Model Scenarios ---

1. Model trained on Body Text, evaluated on Body Text:
  Accuracy: 0.9874
  F1-Score: 0.9874
  Precision: 0.9874
  Recall: 0.9874

2. Model trained on Body Text, evaluated on Title Features:
  F1-Score: 0.4137

3. Model trained on Title Features, evaluated on Title Features:
  Accuracy: 0.9475
  F1-Score: 0.9475
  Precision: 0.9476
  Recall: 0.9475
